# 🚀 歡迎來到你的 ADK 冒險之旅 - 工具與記憶！🚀

歡迎，Agent 架構師！這本筆記本是你賦予 AI agent 兩項關鍵超能力的指南：自訂工具與對話記憶。

在這趟冒險結束時，你將能夠：

- **建構基礎 Agent**：使用 Google Agent Development Kit (ADK) 從零開始建立一個簡單但有效的 AI agent。

- **透過自訂工具賦予新技能**：藉由將 agent 連接到外部 API（例如即時天氣服務），教它執行新任務。

- **建立 Agent 團隊**：組建一個多 agent 系統，其中主要 agent 可以將專門任務委派給其他 agent。

- **精通對話記憶**：了解 Session 在使 agent 能夠記住先前互動、處理回饋，以及進行連貫對話中所扮演的關鍵角色。


讓我們開始這趟冒險吧！

```
  (\__/)
  (•ㅅ•)
  /づ  📚      享受學習 AI Agents 的樂趣 :)
```


-------------
### 🎁 🛑 重要前置條件：設定你的環境！🛑 🎁
-----------------------------------------------------------------------------

👉 **在這裡取得你的 API Key**：https://codelabs.developers.google.com/onramp/instructions#1

 -----------------------------------------------------------------------------

```
 ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️
   /\_/\     /\_/\     /\_/\      /\_/\       /\_/\
  ( ^_^ )   ( -.- )   ( >_< )   ( =^.^= )    ( o_o )             
```


> ### 📌 關於 ADK 2.0
>
> 這本教的東西（`FunctionTool`、`AgentTool`、`google_search`、`ToolContext`、
> `Runner`、session 記憶）在 **ADK 2.0 完全沒有 deprecated**，API 一模一樣，
> 所以這本不需要改寫，直接就是 2.0 相容的。
>
> 被 ADK 2.0 deprecated 的是 **`SequentialAgent` / `LoopAgent` / `ParallelAgent`**
> 這三個編排類別（改用 `Workflow`）。那部分請看同資料夾的
> **`ADK_Learning_multi_agents_ADK2.ipynb`**，裡面有完整的遷移對照表。
>
> 本檔已做的小修：`PROJECT_ID` 改讀環境變數（不再寫死）、更新一處過時註解、
> 最終回覆抽取加防呆。

## 第 0 部分：設定與驗證 🔑

首先，讓我們準備好所有工具。這個步驟會安裝必要的函式庫，並安全地設定你的 Google API key，讓你的 agent 能夠存取 Gemini 的強大功能。

In [ ]:
# === 套件安裝（首次執行時取消註解）===
# google-adk         : Google Agent Development Kit，本 notebook 主角
# google-generativeai: Gemini API 的 Python SDK（ADK 底層依賴）
# plotly             : 互動式圖表（後面評估指標視覺化會用）
# !uv pip install google-adk google-generativeai -q
# !uv pip install plotly -q


In [ ]:
# === 匯入所有必要的函式庫 ===
# 這格把 ADK 開發會用到的所有模組一次載完，方便後續 cell 直接使用。

# --- 標準函式庫：系統操作、非同步、資料處理 ---
import os                          # 環境變數與作業系統介面（讀取 API key、路徑等）
import sys                         # 系統相關參數（如 sys.path、stdout）
import json                        # JSON 解析與序列化（處理 agent 輸入輸出）
import asyncio                     # 非同步事件迴圈（ADK Runner 完全 async-first）
import random                      # 亂數生成（測試資料、隨機選擇）
import string                      # 字串常數（生成隨機 ID 用）
from uuid import uuid4             # 產生唯一 session_id / user_id
from typing import Any, List       # 型別註解（提升程式碼可讀性與 IDE 支援）

# --- 資料分析與視覺化 ---
import pandas as pd                # 資料表格處理（評估結果分析、log 整理）
import plotly.graph_objects as go  # 互動式圖表（視覺化 agent 評估指標）

# --- Google Cloud Vertex AI ---
import vertexai                    # Vertex AI 客戶端（連接 Google Cloud 上的 Gemini 模型）
# from google.colab import auth    # Colab 認證（本機執行不需要，故註解）

# --- Notebook 顯示工具（在 Jupyter / Colab 中渲染結果）---
from IPython.display import HTML, Markdown, display

# === ADK（Agent Development Kit）核心元件 ===
# 這四個是 ADK 開發的「四大金剛」，理解它們等於理解 ADK 80%
from google.adk.agents import Agent              # Agent 主類別：宣告式定義一個 LLM agent（name + instruction + tools）
from google.adk.events import Event              # Event：agent 執行過程中的串流事件（thinking / tool_call / final_response 等）
from google.adk.runners import Runner            # Runner：實際執行 agent 的引擎，管理 session + event loop
import google.adk as adk                         # ADK 模組別名（方便存取其他子模組）
from google.adk.tools import google_search       # 內建 tool：讓 agent 能呼叫 Google 搜尋（grounded answer 必備）
from google.adk.sessions import InMemorySessionService, Session
                                                  # SessionService：管理對話歷史
                                                  # InMemory 版適合開發；生產環境用 Firestore/Redis 版本

# === Gemini 原生型別（用於建構 agent 輸入內容）===
from google.genai import types                   # types 命名空間（包含各種 Gemini API 結構）
from google.genai.types import Content, Part     # Content/Part：Gemini 訊息的標準格式
                                                 # 一則訊息 = Content(role="user", parts=[Part(text=...), Part(image=...)])
                                                 # parts 設計成 list 是為了支援多模態（文字 + 圖片 + 音訊）

print("✅ 所有函式庫已就緒，可以開始建立 agent！")

# ── 安靜的 log ───────────────────────────────────────────────────────────
# google-genai 每次 LLM 呼叫都會用 logging 印一段「AFC 用法建議」的長文，
# 在 notebook 裡會把真正的輸出洗掉。只關這個 logger，不要用光禿禿的
# warnings.filterwarnings("ignore")（那會把整個 kernel 之後的警告全部靜音）。
import logging
import warnings
# ADK 的部分功能還標著 experimental，建立 FunctionTool 宣告時會噴 UserWarning
warnings.filterwarnings("ignore", message=r"\[EXPERIMENTAL\].*")
logging.getLogger("google_genai").setLevel(logging.ERROR)
logging.getLogger("google_adk").setLevel(logging.ERROR)


### 驗證並設定你的專案
要使用 Vertex AI，你需要一個有效的 Google Cloud 專案。本節負責驗證你的環境並設定必要的專案配置。

In [ ]:
# === Colab 認證（本機 Jupyter 不需要，故全部註解）===
# 在 Colab 跑時：取消註解，會跳出 Google 登入視窗，授權後 ADC (Application Default Credentials) 自動就緒
# 在本機跑時：用 `gcloud auth application-default login`（見下方 cell 15）

# # ---  Authentication & Project Configuration ---

# # Authenticate user in Colab
# if "google.colab" in sys.modules:
#     auth.authenticate_user()
#     print("✅ Authenticated successfully.")


✅ Authenticated successfully.


In [ ]:
# === 設定 Google Cloud Project + Vertex AI ===
# ADK 透過 google-genai 呼叫 Gemini。走哪一條由環境變數決定：
#   GOOGLE_GENAI_USE_VERTEXAI = "True" → Vertex AI（企業認證、配額共享、Cloud Logging 整合）
#   GOOGLE_GENAI_USE_VERTEXAI = "0"    → AI Studio（只要一把 API key，門檻低很多）
# 這本走 Vertex AI，需要：GOOGLE_CLOUD_PROJECT / GOOGLE_CLOUD_LOCATION 兩個變數 + ADC 認證。
#
# ⚠️ PROJECT_ID 不寫死在教材裡（寫死的話教材發出去就等於把自己的專案 ID 一起發出去）。
#    優先讀環境變數，沒有才問。

import os


def can_prompt() -> bool:
    """有沒有前端可以回答輸入？JupyterLab / Colab → True；nbclient 這種 headless → False。"""
    try:
        return bool(get_ipython().kernel._allow_stdin)  # noqa: F821
    except Exception:
        return False


PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")

if not PROJECT_ID and can_prompt():
    PROJECT_ID = input("請輸入你的 GCP PROJECT_ID: ").strip()

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "True")

# 首次設定才要啟用 API（每次都跑很慢又吵，所以註解掉；沒啟用過就把它打開跑一次）
# !gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}

# 認證判斷要看實際模式，不然「AI Studio 明明能跑卻印 ❌」會誤導人
VERTEX_MODE = str(os.environ.get("GOOGLE_GENAI_USE_VERTEXAI", "True")).lower() in ("1", "true", "yes")
ADC = os.path.expanduser("~/.config/gcloud/application_default_credentials.json")

if VERTEX_MODE:
    HAS_AUTH = bool(PROJECT_ID) and (os.path.isfile(ADC)
                                     or bool(os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")))
    print(f"認證模式  Vertex AI")
    print(f"專案      {PROJECT_ID or '❌ 未設定'}")
    print(f"區域      {LOCATION}")
    print(f"ADC       {'✅ 已就緒' if os.path.isfile(ADC) else '❌ 找不到'}")
else:
    HAS_AUTH = bool(os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY"))
    print(f"認證模式  AI Studio API key")
    print(f"API key   {'✅ 已設定' if HAS_AUTH else '❌ 未設定'}")

print(f"可以呼叫 Gemini 了嗎：{'✅ 可以' if HAS_AUTH else '❌ 還沒'}")

if not HAS_AUTH and VERTEX_MODE:
    print("\n⚠️ Vertex AI 還沒準備好，請執行（見下面 cell 15）：")
    print("   gcloud auth application-default login --project=<你的專案ID>")
    print("   gcloud services enable aiplatform.googleapis.com --project=<你的專案ID>")
    print("\n   或改用 AI Studio（門檻低很多）：")
    print('   os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "0"')
    print('   os.environ["GOOGLE_API_KEY"] = "<你的 key>"   # https://aistudio.google.com/apikey')


---
## 第 1 部分：你的第一個 Agent - 一日遊精靈 🧞

來認識你的第一個作品！`day_trip_agent` 是一個簡單但強大的助手。我們透過教它理解**預算限制**來讓它更聰明。

* **Agent**：運作的核心，由其指令、工具和使用的 AI 模型所定義。
* **Session**：對話歷史。對於這個簡單的 agent，它只是一個單一請求-回應的容器。
* **Runner**：連接 `Agent` 和 `Session` 的引擎，用來處理你的請求並取得回應。

```
+--------------------------------------------------+
|         隨興一日遊 Agent 🤖                       |
|--------------------------------------------------|
|  模型：gemini-2.5-flash                          |
|  描述：                                          |
|   根據心情、興趣和預算生成全天行程                 |
|--------------------------------------------------|
|  🔧 工具：                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 能力：                                       |
|   - 預算意識（節省 / 奢華）                       |
|   - 心情匹配（冒險、放鬆等）                      |
|   - 即時資訊（營業時間、活動）                     |
|   - 早上 / 下午 / 晚上規劃                        |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    |   使用者輸入      |
    |------------------|
    |  心情             |
    |  興趣             |
    |  預算             |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|             輸出：Markdown 行程表                 |
|--------------------------------------------------|
| - 時間區塊（早上 / 下午 / 晚上）                  |
| - 含連結和營業時間的場所名稱                       |
| - 符合預算的活動                                  |
+--------------------------------------------------+
```


In [4]:
# === Agent 定義：Day Trip Genie（一日遊行程生成器）===
# 這是最簡單的 ADK agent 結構 — 純 LLM + 內建工具（google_search）
# 沒有自定義 function tool，靠 instruction 引導 LLM 行為

def create_day_trip_agent():
    """建立 Spontaneous Day Trip Generator agent

    用 factory function 而不是直接 inst 化的好處：
    - 每次呼叫都產生全新 instance（避免 instance state 殘留問題）
    - 方便單元測試 mock
    - 方便當成 Workflow 的節點重複使用
      （ADK 2.0 起 SequentialAgent / ParallelAgent 已 deprecated，改用 Workflow；
       見同資料夾的 ADK_Learning_multi_agents_ADK2.ipynb）
    """
    return Agent(
        # === 必填四元素 ===
        name="day_trip_agent",                    # 唯一名稱（會出現在 log/event 中，用來識別是哪個 agent 在動）
        model="gemini-2.5-flash",                 # 用哪個 Gemini 模型；2.5-flash 速度快成本低，pro 比較強但慢
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
                                                   # description 給「其他 agent」看的（hierarchical agent 中用來路由）
                                                   # 不是給 LLM 自己看的
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).
        5. **translator**: translate the itinerary to the user's language.

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        # instruction 是 system prompt，會在每次對話時附在訊息最前面
        # 寫法建議：角色 → 任務 → 規則 → 輸出格式（這樣 LLM 比較不會漏）

        tools=[google_search]                      # 內建 tool；agent 會自己決定何時呼叫
                                                    # 相對於 OpenAI 的 function calling，ADK 把流程包得更高階
    )

# 真正建立 agent instance
day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")


🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [ ]:
# === Helper：執行 agent 查詢的通用函式 ===
# 這個 function 整個 notebook 都會用，封裝了「建 Runner → 執行 → 收 events → 抽 final response」的流程
# 學會這個 pattern 等於學會 ADK 90% 的執行模型

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """初始化 Runner 並執行查詢

    Args:
        agent     : 要執行的 Agent 實例
        query     : 使用者的問題（純文字）
        session   : 對話 session（決定 agent 是否「記得」之前的對話）
        user_id   : 使用者 ID（多租戶系統用，這裡單機所以隨便設）
        is_router : True 時不印 event 細節（router 場景不關心中間過程）
    """
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    # === Runner：ADK 的執行引擎 ===
    # 每次查詢都建一個新 Runner（成本很低，不用快取）
    # session_service 是全域共用的（後面會建立）→ 多個 Runner 共用同一個 session store
    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name                    # app_name 是 session 的 namespace，通常用 agent name 就好
    )

    final_response = ""
    try:
        # === run_async 是 async generator，串流回傳 events ===
        # event 類型可能有：thinking、tool_call、tool_response、final_response 等
        # 用 async for 邊跑邊收，可以做即時 UI 更新（streaming chat）
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            # Content/Part 是 Gemini 的訊息格式，role="user" 表示這是使用者輸入
            # 多模態時可以塞 [Part(text=...), Part(image=...)]
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # 印每個 event，可以觀察 agent 內部「思考」與「呼叫工具」的過程
                # 學習階段強烈建議印出來看，理解 agent 怎麼跑
                print(f"EVENT: {event}")
            # is_final_response() = True 才是最終答案；其他是中間過程
            if event.is_final_response():
                # 防呆：parts 可能是空的，parts[0].text 也可能是 None
                # （例如模型只回了 tool call、或安全機制擋掉），直接索引會 IndexError
                parts = event.content.parts if event.content else None
                if parts:
                    final_response = " ".join(p.text for p in parts if p.text) or final_response
    except Exception as e:
        # 任何錯誤都包成字串回傳，避免整個 notebook 中斷
        # 正式環境應該分類處理（API 限流、network、模型 refusal 等）
        final_response = f"An error occurred: {e}"

    if not is_router:
     # 用 Markdown 渲染 → agent 輸出有 markdown 時才會好看
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# === 全域 Session Service：所有 cell 共用 ===
# InMemorySessionService 把對話歷史存在 Python 記憶體
# kernel restart 後資料會消失 → 開發測試 OK；正式用 VertexAiSessionService / 自定 backend
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"   # 使用者 ID；多租戶系統會用真實 user UUID


In [ ]:
# === 安裝 gcloud CLI（macOS）===
# 如果你還沒裝 gcloud CLI，取消註解執行這行
# Linux/Windows 請參考官方文件 https://cloud.google.com/sdk/docs/install
# !brew install google-cloud-sdk


==> Auto-updating Homebrew...
Adjust how often this is run with `$HOMEBREW_AUTO_UPDATE_SECS` or disable with
`$HOMEBREW_NO_AUTO_UPDATE=1`. Hide these hints with `$HOMEBREW_NO_ENV_HINTS=1` (see `man brew`).
==> Downloading https://ghcr.io/v2/homebrew/core/portable-ruby/blobs/sha256:f41c72b891c40623f9d5cd2135f58a1b8a5c014ae04149888289409316276c72
######################################################################### 100.0% 18.8%           56.1%
==> Pouring portable-ruby-4.0.2_1.arm64_big_sur.bottle.tar.gz
==> Auto-updated Homebrew!
Updated 2 taps (steipete/tap and mongodb/brew).

You have 117 outdated formulae and 6 outdated casks installed.

==> Fetching downloads for: gcloud-cli
⠋ API Source gcloud-cli.rb
⠋ Cask gcloud-cli (564.0.0)⠋ API Source gcloud-cli.rb
⠋ Cask gcloud-cli (564.0.0)✔︎ API Source gcloud-cli.rb                           Verified      3.8KB/  3.8KB
⠙ Cask gcloud-cli (564.0.0)                          Downloading  86.0KB/ 59.4MB⠙ Cask gcloud-cli (564.0.0)            

In [ ]:
# === 本機 Application Default Credentials 認證 ===
# 跑這格會：
#   1. 開瀏覽器跳出 Google 登入頁
#   2. 你授權後，憑證存到 ~/.config/gcloud/application_default_credentials.json
#   3. 之後 google.adk / google.genai 會自動讀這份憑證連 Vertex AI
# 在 Colab 跑請改用 cell 8 的 colab.auth.authenticate_user()
# !gcloud auth application-default login


Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=xHIKgHIZeTny9fyn9b8ADH2uXhA9fR&access_type=offline&code_challenge=uvHxva-flJA2Rya6GLWRr9ZMLh7WyumIyEN7IRatxLc&code_challenge_method=S256


Credentials saved to file: [/Users/kevinluo/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).
Cannot add the project "real-hotai-production" to ADC as the quota project because the account in ADC does not have the "serviceusage.services.use" permission on this project. You might receive a "quota_exceeded" or "API not enabled" erro

In [6]:
# === 測試 Day Trip Genie：台中文青一日遊 ===

async def run_day_trip_genie():
    # 為這次查詢建一個全新 session（一次性對話，不需要記憶）
    # 對比 cell 31 的多輪對話會重複用同一個 session
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # 注意 query 中的「經濟實惠」→ instruction 裡有 budget-aware 規則會被觸發
    query = "請幫我規劃一趟台灣台中，近郊輕鬆且充滿藝術（文青）氣息的一日遊，行程請盡量經濟實惠。"
    print(f"🗣️ User Query: '{query}'")

    # await 因為這個 helper 是 async function；Jupyter 內 cell 頂層可以直接 await
    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()


🗣️ User Query: '請幫我規劃一趟台灣台中，近郊輕鬆且充滿藝術（文青）氣息的一日遊，行程請盡量經濟實惠。'

🚀 Running query for agent: 'day_trip_agent' in session: '04497ac6-9bcd-476e-bc05-e5e4f4e61869'...


/Users/kevinluo/google-agent-ecosystem/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""為您規劃一趟台中霧峰近郊，兼具輕鬆、藝術（文青）氣息且經濟實惠的一日遊行程，讓您在探索文創聚落與建築之美的同時，也能享受悠閒時光並控制預算。

---

### **台中霧峰｜近郊輕鬆文青一日遊（經濟實惠版）**

**主題：** 探索老眷村文創聚落與現代建築藝術，享受悠閒時光。

**預算：** 低至中等（主要花費為交通、午晚餐及個人購物）。

#### **上午：懷舊眷村文創巡禮 (9:30 AM - 12:30 PM)**

*   **光復新村 (Guangfu New Village)**
    *   **活動內容：** 漫步於充滿歷史感的眷村巷弄，這裡曾是台灣第一個新市鎮，現在搖身一變成為文創基地。您可以探索各式特色小店、獨立咖啡廳、手作工坊和藝術裝置。紅磚牆、老式建築與綠意盎然的環境，非常適合拍照打卡，感受「文青」氛圍。無需門票，可自由參觀。
    *   **特色：** 免費入場，充滿藝術氣息與懷舊感，商店營業時間各異，但整體空間開放參觀。
    *   **地址：** 台中市霧峰區光復新村 (確切位置通常為霧峰區光復新站和平路2-3號周邊)
    *   **開放時間：** 全天候開放，各店家營業時間不同。建議白天前往，約09:30後店家陸續開門。

#### **午餐：品嚐霧峰在地美食 (12:30 PM - 1:30 PM)**

*   **活動內容：** 在霧峰光復新村周邊或霧峰市區尋找經濟實惠的在地小吃。霧峰以爌肉飯、肉圓、麵食等傳統美食聞名，許多店家提供CP值高的餐點。
*   **推薦選擇：** 尋找當地受歡迎的麵攤、肉羹店或小吃店，體驗道地的台灣風味，價格通常十分親民。

#### **下午：安藤忠雄建築與現代藝術饗宴 (1:30 PM - 4:30 PM)**

*   **亞洲大學現代美術館 (Asia University Museum of Modern Art)**
    *   **活動內容：** 前往由普立茲克建築獎得主安藤忠雄設計的亞洲大學現代美術館。建築本身就是一件藝術品，以清水模與特殊幾何三角形為設計元素，光是欣賞建築外觀

為您規劃一趟台中霧峰近郊，兼具輕鬆、藝術（文青）氣息且經濟實惠的一日遊行程，讓您在探索文創聚落與建築之美的同時，也能享受悠閒時光並控制預算。

---

### **台中霧峰｜近郊輕鬆文青一日遊（經濟實惠版）**

**主題：** 探索老眷村文創聚落與現代建築藝術，享受悠閒時光。

**預算：** 低至中等（主要花費為交通、午晚餐及個人購物）。

#### **上午：懷舊眷村文創巡禮 (9:30 AM - 12:30 PM)**

*   **光復新村 (Guangfu New Village)**
    *   **活動內容：** 漫步於充滿歷史感的眷村巷弄，這裡曾是台灣第一個新市鎮，現在搖身一變成為文創基地。您可以探索各式特色小店、獨立咖啡廳、手作工坊和藝術裝置。紅磚牆、老式建築與綠意盎然的環境，非常適合拍照打卡，感受「文青」氛圍。無需門票，可自由參觀。
    *   **特色：** 免費入場，充滿藝術氣息與懷舊感，商店營業時間各異，但整體空間開放參觀。
    *   **地址：** 台中市霧峰區光復新村 (確切位置通常為霧峰區光復新站和平路2-3號周邊)
    *   **開放時間：** 全天候開放，各店家營業時間不同。建議白天前往，約09:30後店家陸續開門。

#### **午餐：品嚐霧峰在地美食 (12:30 PM - 1:30 PM)**

*   **活動內容：** 在霧峰光復新村周邊或霧峰市區尋找經濟實惠的在地小吃。霧峰以爌肉飯、肉圓、麵食等傳統美食聞名，許多店家提供CP值高的餐點。
*   **推薦選擇：** 尋找當地受歡迎的麵攤、肉羹店或小吃店，體驗道地的台灣風味，價格通常十分親民。

#### **下午：安藤忠雄建築與現代藝術饗宴 (1:30 PM - 4:30 PM)**

*   **亞洲大學現代美術館 (Asia University Museum of Modern Art)**
    *   **活動內容：** 前往由普立茲克建築獎得主安藤忠雄設計的亞洲大學現代美術館。建築本身就是一件藝術品，以清水模與特殊幾何三角形為設計元素，光是欣賞建築外觀與內部光影變化就已值回票價。館內不定期舉辦各種現代藝術展覽。
    *   **費用：** 全票 NT$250。
    *   **開放時間：** 週二至週日 09:30 - 17:00 (週一休館)。
    *   **地址：** 台中市霧峰區柳豐路500號 (亞洲大學校園內)
    *   **經濟實惠小撇步：** 如果預算非常緊縮，亦可選擇在亞洲大學校園內漫步，欣賞安藤忠雄建築的外觀，感受其獨特的設計美學，而不入內參觀展覽，同樣能體驗到藝術氛圍。

#### **傍晚：悠閒散步與晚餐 (4:30 PM - 6:00 PM)**

*   **活動內容：** 在霧峰區或返回光復新村周邊，找一家舒適的咖啡廳小憩片刻，或在附近公園散步，感受夕陽下的悠閒。
*   **晚餐選擇：** 依個人喜好，再次選擇霧峰在地小吃，或是前往光復新村內提供輕食、簡餐的文創小店享用晚餐。

---

#### **交通建議：**

*   **大眾運輸：** 從台中市區可搭乘公車前往霧峰光復新村，車程約40分鐘至1小時，在「坑口里」站下車步行即可抵達。從光復新村前往亞洲大學現代美術館可搭乘公車或計程車，車程不遠。
*   **自行開車/騎車：** 若自行開車或騎車，可將車停在光復新村周邊的停車場，前往亞洲大學現代美術館亦方便。

這趟行程結合了歷史聚落的懷舊美、現代建築的設計感與在地美食的滋味，讓您能在輕鬆的步調中，感受台中近郊獨特的文青魅力！

--------------------------------------------------



In [19]:
# === 測試 Day Trip Genie：全台範圍版本 ===
# 與上一格的差異：去掉「台中」限制，讓 agent 自己選地點
# 這時 google_search 會發揮更大作用 — agent 需要先想「台灣有哪些文青景點」

async def run_day_trip_genie():
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    query = "請幫我規劃一趟台灣近郊輕鬆且充滿藝術（文青）氣息的一日遊，行程請盡量經濟實惠。"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()


🗣️ User Query: '請幫我規劃一趟台灣近郊輕鬆且充滿藝術（文青）氣息的一日遊，行程請盡量經濟實惠。'

🚀 Running query for agent: 'day_trip_agent' in session: 'b0b48d3e-3536-4ea6-b550-0ce5d20e3381'...


/Users/kevinluo/google-agent-ecosystem/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""好的，這就為您規劃一趟台灣近郊輕鬆、充滿藝術（文青）氣息且經濟實惠的一日遊，地點選定以陶瓷藝術聞名的新北市鶯歌區。鶯歌不僅有豐富的陶藝文化，交通也相當便利，非常適合一日輕旅行。

---

### **【鶯歌文青陶藝一日遊：經濟實惠輕旅行】**

**💡 旅行主題：** 陶藝文化、文創探索、悠閒咖啡時光

**💰 預算考量：** 以大眾交通工具為主，餐飲選擇當地平價小吃，博物館門票經濟實惠或可享優惠。

---

#### **上午時光 (09:30 - 12:30)：陶瓷藝術的殿堂巡禮**

*   **09:30 抵達鶯歌 + 步行前往陶瓷博物館**
    搭乘火車至**鶯歌車站**。從車站出發，您可以選擇步行約10-15分鐘，沿途感受小鎮風光，或搭乘短程公車前往「新北市立鶯歌陶瓷博物館」。 火車是從台北市區前往鶯歌最經濟且便捷的方式之一。
*   **10:00 - 12:30 新北市立鶯歌陶瓷博物館**
    開始您的藝術之旅！新北市立鶯歌陶瓷博物館是台灣第一座以陶瓷為主題的專業博物館，建築本身採用清水模和鋼骨結構，充滿現代感與藝術氛圍，非常適合拍照打卡。館內展示了台灣陶瓷的歷史發展、製作過程及當代陶藝作品，常設展與特展內容豐富，讓您深入了解陶瓷文化之美。部分展區提供互動體驗，增添趣味。
    *   **費用：** 全票新台幣80元。若您設籍新北市、年滿65歲以上、未滿12歲、在學學生、身心障礙者及其一位必要陪伴者，皆可享免門票優惠。
    *   **開放時間：** 平日09:30-17:00；假日09:30-18:00 (每月第一個星期一休館)。

#### **午間時光 (12:30 - 13:30)：品嚐在地古早味**

*   **12:30 - 13:30 鶯歌老街周邊平價午餐**
    從陶瓷博物館步行約5-10分鐘即可抵達熱鬧的**鶯歌陶瓷老街**。 老街上有許多經濟實惠的在地美食可供選擇。您可以嘗試：
    *   **厚道飲食店：** 以懷舊風格裝潢，提供炸排骨飯等古早味飯食，份量足、口味好，深受在地人喜愛。
    *   **阿婆壽司：** 

好的，這就為您規劃一趟台灣近郊輕鬆、充滿藝術（文青）氣息且經濟實惠的一日遊，地點選定以陶瓷藝術聞名的新北市鶯歌區。鶯歌不僅有豐富的陶藝文化，交通也相當便利，非常適合一日輕旅行。

---

### **【鶯歌文青陶藝一日遊：經濟實惠輕旅行】**

**💡 旅行主題：** 陶藝文化、文創探索、悠閒咖啡時光

**💰 預算考量：** 以大眾交通工具為主，餐飲選擇當地平價小吃，博物館門票經濟實惠或可享優惠。

---

#### **上午時光 (09:30 - 12:30)：陶瓷藝術的殿堂巡禮**

*   **09:30 抵達鶯歌 + 步行前往陶瓷博物館**
    搭乘火車至**鶯歌車站**。從車站出發，您可以選擇步行約10-15分鐘，沿途感受小鎮風光，或搭乘短程公車前往「新北市立鶯歌陶瓷博物館」。 火車是從台北市區前往鶯歌最經濟且便捷的方式之一。
*   **10:00 - 12:30 新北市立鶯歌陶瓷博物館**
    開始您的藝術之旅！新北市立鶯歌陶瓷博物館是台灣第一座以陶瓷為主題的專業博物館，建築本身採用清水模和鋼骨結構，充滿現代感與藝術氛圍，非常適合拍照打卡。館內展示了台灣陶瓷的歷史發展、製作過程及當代陶藝作品，常設展與特展內容豐富，讓您深入了解陶瓷文化之美。部分展區提供互動體驗，增添趣味。
    *   **費用：** 全票新台幣80元。若您設籍新北市、年滿65歲以上、未滿12歲、在學學生、身心障礙者及其一位必要陪伴者，皆可享免門票優惠。
    *   **開放時間：** 平日09:30-17:00；假日09:30-18:00 (每月第一個星期一休館)。

#### **午間時光 (12:30 - 13:30)：品嚐在地古早味**

*   **12:30 - 13:30 鶯歌老街周邊平價午餐**
    從陶瓷博物館步行約5-10分鐘即可抵達熱鬧的**鶯歌陶瓷老街**。 老街上有許多經濟實惠的在地美食可供選擇。您可以嘗試：
    *   **厚道飲食店：** 以懷舊風格裝潢，提供炸排骨飯等古早味飯食，份量足、口味好，深受在地人喜愛。
    *   **阿婆壽司：** 鶯歌的超人氣老字號小吃，提供多種口味的壽司、涼麵、蒸蛋和味噌湯，且24小時營業，價格親民。
    *   **勇伯垃圾麵：** 鶯歌在地飄香五十年的特色麵食，雖然名字特別，但其獨特風味值得一試。

#### **下午時光 (13:30 - 17:00)：文創老街漫遊與咖啡香**

*   **13:30 - 15:30 鶯歌陶瓷老街深度漫遊**
    在老街上盡情探索！鶯歌陶瓷老街不同於其他老街，以「陶」為主題，濃厚的藝術氣息非常適合陶冶身心。您可以逛逛各式陶瓷專賣店、陶藝工作室，欣賞獨特的陶瓷藝術品，甚至觀察匠師們的創作過程。老街內也有許多文創小店，可以找到獨特的紀念品或手作小物。 假日時段（平日中午12點到下午5點以及假日全天）老街會成為人行徒步區，逛起來更為舒適。
*   **15:30 - 17:00 文青咖啡廳悠閒小憩**
    逛累了，找間充滿文藝氣息的咖啡廳歇歇腳。鶯歌老街周邊有不少風格獨特的咖啡廳：
    *   **日知常屋：** 位於老街內的日式文青咖啡廳，採光優美、環境舒適，提供咖啡茶飲及點心。
    *   **去哪咖啡 (Kido Cafe)：** 藏身於鶯歌市場內，距離火車站約10分鐘步行路程，以咖啡本身為重點，風格獨特。
    *   **老咖Cafe：** 位於鶯歌車站周圍，步行約五分鐘，店內風格歐式復古，除了咖啡也有鬆餅等餐點選擇。
    點一杯咖啡或茶，享受悠閒的午後時光，感受鶯歌的慢活步調。

#### **傍晚時光 (17:00 - 18:00)：伴手禮與歸途**

*   **17:00 - 17:30 購買特色伴手禮**
    在老街上挑選一些獨特的陶瓷工藝品、文創商品作為紀念，或者購買一些鶯歌在地特色小吃，如「阿嬤ㄟ豆花」或包子等。
*   **17:30 - 18:00 前往鶯歌車站，賦歸**
    帶著滿滿的藝術收穫和愉悅心情，步行或搭乘公車返回鶯歌車站，結束這趟經濟實惠的文青一日遊。

---

**交通方式建議：**
從台北車站搭乘台鐵區間車至鶯歌車站，車程約25-35分鐘，班次密集，是最推薦的交通方式。

**小提醒：**
*   建議穿著輕便舒適的鞋子，因為行程中會包含不少步行。
*   每月的第一個星期一為鶯歌陶瓷博物館的休館日，請避開此日期前往。
*   雖然老街大部分店家營業至傍晚，但部分店家可能會提早打烊，建議提早前往。

--------------------------------------------------



---
## 第 2 部分：用自訂工具為 Agent 增強能力 🛠️

到目前為止，我們使用了強大的內建 `GoogleSearch` 工具。但 agent 的真正威力來自於將它們連接到你自己的邏輯和資料來源。

這就是**自訂工具**發揮作用的地方。讓我們用實際的應用範例來探索三種賦予 agent 新技能的模式。

### 2.1 簡單的 `FunctionTool`：呼叫即時天氣 API

建立工具最直接的方式是撰寫一個 Python 函式。這非常適合同步任務，例如從 API 取得資料。

**核心概念：** 函式的 **docstring** 至關重要。ADK 將其用作工具的官方描述，LLM 會讀取它來了解工具的用途、參數以及何時使用它。

在這個範例中，我們將建立一個呼叫**免費、公開的美國國家氣象局 API** 的工具來取得即時天氣預報。不需要 API key！

In [20]:
# === Tool 定義：呼叫真實天氣 API ===
# 這是「自定義 FunctionTool」的範例 — 用 Python function 包住外部 API
# ADK 的魔法：你寫一個普通 function，丟到 Agent(tools=[...]) 就能用
# 它會自動從 docstring 抽出 description / args，告訴 LLM 何時呼叫

import requests
import json

# 簡化版 geocoding（避免另外引入 geocoding API）
# 正式專案應該用 Google Maps API 或 OpenStreetMap Nominatim
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324"
}

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in the US.

    Args:
        location: The city name, e.g., "San Francisco".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    # ⚠️ docstring 超重要！ADK 會直接用它告訴 LLM 「這個 tool 是做什麼、何時用」
    #    寫得越清楚，LLM 越不會誤呼叫

    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")
    # 印出 tool call → 開發時觀察 agent 何時呼叫了 tool（debugging 必備）

    # === 步驟 1: 把地名轉座標（geocoding）===
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        # 回傳 dict 含 "status": "error"，LLM 看得懂錯誤、可以告訴使用者
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # === 步驟 2: 呼叫 NWS（美國國家氣象局）API — 兩段式呼叫 ===
        # 第一段：用座標查 forecast URL（NWS 把不同 grid 的預報拆到不同 endpoint）
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}  # NWS 規定要帶 User-Agent
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status()  # 4xx/5xx 直接拋例外
        forecast_url = points_response.json()['properties']['forecast']

        # 第二段：抓真正的 forecast
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # 取最近一個時段的預報
        current_period = forecast_response.json()['properties']['periods'][0]
        # 回傳結構化 dict → LLM 比較容易引用各個欄位
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}

# === Agent 定義：使用上方 tool 的 agent ===
weather_agent = Agent(
    name="weather_aware_planner",
    model="gemini-2.5-flash",
    description="A trip planner that checks the real-time weather before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    # instruction 用「MUST use」強調 → 提升 LLM 呼叫 tool 的機率
    # 不寫的話 LLM 可能會用內部知識瞎猜天氣，不會主動呼叫 tool

    tools=[get_live_weather_forecast]
    # tools 直接傳 Python function 即可；ADK 會自動包成 FunctionTool
    # 也可以手動 from google.adk.tools import FunctionTool 包裝（高級用法可加 metadata）
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")


🌦️ Agent 'weather_aware_planner' is created and can now call a live weather API!


In [23]:
# === 測試 Weather-Aware Planner ===
# 期待行為：agent 收到「爬山」query → 觸發「outdoor activity」規則 → 呼叫 weather tool → 整合結果

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    query = "我想在美國舊金山進行爬山之旅，天氣怎麼樣？"
    print(f"🗣️ User Query: '{query}'")
    # 觀察 EVENT log → 應該會看到 tool_call 事件 → tool_response 事件 → final_response
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()


🗣️ User Query: '我想在美國舊金山進行爬山之旅，天氣怎麼樣？'

🚀 Running query for agent: 'weather_aware_planner' in session: 'fcfc219b-253c-4237-82dd-d94cfede1a8d'...


/Users/kevinluo/google-agent-ecosystem/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'location': 'San Francisco'
        },
        id='adk-5d4575f0-5a5d-4055-93d7-c0a4be71cb0a',
        name='get_live_weather_forecast'
      ),
      thought_signature=b'\n\xe5\x01\x01\x8f=k_\xdex\xaa\x85\xe4\xa5 \xf2\xb0\xbc\xec\x1fR\xf4\xeb\xcc\xab\xae\xc1\xf0\xa7\xa6\x1f\xb8\x13\x90\x02g$\xaf\x7fO\x1b\xf1\n\x9e\xfddZQ\xdb6fy\xb6\xc4s\xe9!\x0c\xc5\xa1\x95\xf0d\xcf5`\xf0\xf3\xb7\xb3\\Z\xec\x9b\x91\xde\xfc\xae\x15m\x1c\x05\x11A\xfd,\x83\xbd\xb5\x03\x91\xc0\x9e\xce\x89\xb07...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=10,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
 

/Users/kevinluo/google-agent-ecosystem/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""舊金山目前氣溫約 56°F。清晨 5 點後局部有霧，大部分時間多雲，最低氣溫約 56°F。西南偏西風，風速約每小時 3 英里。

基於這些天氣狀況，目前的天氣對於爬山來說有點涼爽且多雲，清晨還可能有霧。建議您穿著多層衣物，並注意防潮。""",
      thought_signature=b'\n\xd3\x04\x01\x8f=k_Yg\xffl\xef\x1d\x01<\n\xec\xedIb\xf8\xfb5\xae\xc8\xf2\xf3\xe33O\x0c`\xbc\x179\xcf\xc2\x0cnk\xcd\x94\xe1\xa1&\xd1]1@\xf3<\xcf\xbaE\x87\xf3\x9cU\x0e>\x11\x12`\x91\x0f2\x1b\x89\x1f\xed,1t\xc7\x8d<\xc1\xe9\xe1\xc7tB\xd4\x82x\xffn\xa2}\x80\x15\xe3\xccq\xb8\xd2...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=95,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=95
    ),
  ],
  prompt_token_count=252,
  prompt_tokens_details=[
    ModalityTo

舊金山目前氣溫約 56°F。清晨 5 點後局部有霧，大部分時間多雲，最低氣溫約 56°F。西南偏西風，風速約每小時 3 英里。

基於這些天氣狀況，目前的天氣對於爬山來說有點涼爽且多雲，清晨還可能有霧。建議您穿著多層衣物，並注意防潮。

--------------------------------------------------



## 2.2 Agent 即工具：諮詢專家 🧑‍🍳

既然可以建立一個**專家 agent 團隊**，為什麼要建立一個做所有事情的 agent 呢？**Agent 即工具**模式允許一個 agent 將任務委派給另一個 agent。

**核心概念：** 這與 sub-agent 不同。當 Agent A 將 Agent B 作為工具呼叫時，Agent B 的回應會被傳**回 Agent A**。Agent A 然後使用該資訊來形成自己對使用者的最終回應。這是一種從更簡單、專注且可重複使用的 agent 組合複雜行為的強大方式。

### 運作方式

我們的頂層 agent `trip_data_concierge_agent` 扮演**協調者**的角色。它有兩個可用的工具：

1.  `call_db_agent`：一個內部呼叫 `db_agent` 來取得原始資料的函式。
2.  `call_concierge_agent`：一個呼叫 `concierge_agent` 的函式。

`concierge_agent` 本身有一個工具：`food_critic_agent`。

複雜查詢的流程如下：

1.  **使用者**向 `trip_data_concierge_agent` 詢問飯店和附近的餐廳。
2.  **協調者**首先呼叫 `call_db_agent` 來取得飯店資料。
3.  資料被儲存在 `tool_context.state` 中。
4.  **協調者**接著呼叫 `call_concierge_agent`，它會從 context 中取得飯店資料。
5.  `concierge_agent` 收到請求並決定需要使用自己的工具 `food_critic_agent`。
6.  `food_critic_agent` 提供一個風趣的推薦。
7.  `concierge_agent` 取得評論家的回應並禮貌地格式化。
8.  這個最終的精緻回應被回傳給**協調者**，由它呈現給使用者。

                         +-----------------------------------------------------------+
                         |              🧭 旅遊資料禮賓 Agent                       |
                         |-----------------------------------------------------------|
                         |  模型：gemini-2.5-flash                                   |
                         |  描述：                                                   |
                         |   協調資料庫查詢和旅遊推薦                                 |
                         |-----------------------------------------------------------|
                         |  🔧 工具：                                                |
                         |   1. call_db_agent                                        |
                         |   2. call_concierge_agent                                 |
                         +-----------------------------------------------------------+
                                      /                                \
                                     /                                  \
                                    ▼                                    ▼
        +-------------------------------------------+    +---------------------------------------------+
        |         🔧 工具：call_db_agent            |    |       🔧 工具：call_concierge_agent          |
        |-------------------------------------------|    |---------------------------------------------|
        | 呼叫：db_agent                            |    | 呼叫：concierge_agent                        |
        |                                           |    | 使用 db_agent 的資料進行推薦                  |
        +-------------------------------------------+    +---------------------------------------------+
                                |                                          |
                                ▼                                          ▼
       +--------------------------------------------+   +------------------------------------------------+
       |              📦 db_agent                   |   |             🤵 concierge_agent                  |
       |--------------------------------------------|   |------------------------------------------------|
       | 模型：gemini-2.5-flash                     |   | 模型：gemini-2.5-flash                          |
       | 角色：回傳模擬的 JSON 飯店資料              |   | 角色：處理使用者問答的飯店員工                    |
       +--------------------------------------------+   | 工具：                                          |
                                                         |  - food_critic_agent                           |
                                                         +------------------------------------------------+
                                                                                 |
                                                                                 ▼
                                                       +------------------------------------------------+
                                                       |          🍽️ food_critic_agent                  |
                                                       |------------------------------------------------|
                                                       | 模型：gemini-2.5-flash                          |
                                                       | 角色：提供風趣的餐廳推薦                         |
                                                       +------------------------------------------------+


In [7]:
# === Agent-as-Tool 模式：階層式 agent 組合 ===
# 這是 ADK 最強的 pattern 之一 — 把一個 agent 包成 tool 給另一個 agent 用
# 結構：Orchestrator → calls db_agent + concierge_agent → concierge → calls food_critic
#
# 為什麼要這樣設計？
#   - 單一 agent 塞太多責任 → instruction 變雜亂、容易出錯
#   - 拆成多層 → 每個 agent prompt 短、職責清楚、可單獨測試
#   - 類似軟體工程的「單一職責原則」

import asyncio
from google.adk.tools import ToolContext            # ToolContext: tool 內部存取共享 state 的介面
from google.adk.tools.agent_tool import AgentTool   # AgentTool: 把 agent 包成 tool 的關鍵類別

# 假設 db_agent 是個 NL2SQL Agent；這裡用 mock 簡化
db_agent = Agent(
    name="db_agent",
    model="gemini-2.5-flash",
    # 為了示範，instruction 直接讓它回 mock JSON
    # 真實場景：db_agent 會用 SQL tool 查資料庫，這裡省略
    instruction="You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}")

# === 1. 定義專家 agents ===

# Food Critic — 最內層的專家（不需要其他 tool）
food_critic_agent = Agent(
    name="food_critic_agent",
    model="gemini-2.5-flash",
    instruction="You are a snobby but brilliant food critic. You ONLY respond with a single, witty restaurant suggestion near the provided location.",
    # 用「snobby but brilliant」設定人格 → LLM 輸出風格會跟著變
)

# Concierge — 中層；它「擁有」food_critic 當 tool
concierge_agent = Agent(
    name="concierge_agent",
    model="gemini-2.5-flash",
    instruction="You are a five-star hotel concierge. If the user asks for a restaurant recommendation, you MUST use the `food_critic_agent` tool. Present the opinion to the user politely.",
    tools=[AgentTool(agent=food_critic_agent)]
    # AgentTool(agent=...) 把另一個 agent 包成 tool
    # concierge 看 food_critic 就像看一般 function tool 一樣
)


# === 2. Orchestrator 用的 tools（手寫 async function 包 AgentTool）===
# 為什麼不直接 tools=[AgentTool(agent=db_agent), AgentTool(agent=concierge_agent)]？
# 因為我們要「在 tool 之間共享 state」— 把 db 結果存起來給 concierge 用
# 包一層 wrapper 才能存取 ToolContext.state

async def call_db_agent(
    question: str,
    tool_context: ToolContext,    # ⭐ 加上這個參數，ADK 會自動注入 ToolContext
):
    """
    Use this tool FIRST to connect to the database and retrieve a list of places, like hotels or landmarks.
    """
    print("--- TOOL CALL: call_db_agent ---")
    agent_tool = AgentTool(agent=db_agent)
    db_agent_output = await agent_tool.run_async(
        args={"request": question}, tool_context=tool_context
    )
    # 把資料存進 state，下一個 tool 可以讀
    # state 是「跨 tool」共享的 dict，每個 session 獨立
    tool_context.state["retrieved_data"] = db_agent_output
    return db_agent_output


async def call_concierge_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    After getting data with call_db_agent, use this tool to get travel advice, opinions, or recommendations.
    """
    print("--- TOOL CALL: call_concierge_agent ---")
    # 從 state 拿出前一個 tool 存的資料
    input_data = tool_context.state.get("retrieved_data", "No data found.")

    # 把 db 結果塞進 prompt 給 concierge → 這就是「tool chaining」
    question_with_data = f"""
    Context: The database returned the following data: {input_data}

    User's Request: {question}
    """

    agent_tool = AgentTool(agent=concierge_agent)
    concierge_output = await agent_tool.run_async(
        args={"request": question_with_data}, tool_context=tool_context
    )
    return concierge_output


# === 3. 頂層 Orchestrator agent ===
# 這個 agent 不直接執行任務，只負責「規劃流程 + 呼叫子 agent」
# 類似軟體架構中的 Facade / Coordinator

trip_data_concierge_agent = Agent(
    name="trip_data_concierge",
    model="gemini-2.5-flash",
    description="Top-level agent that queries a database for travel data, then calls a concierge agent for recommendations.",
    tools=[call_db_agent, call_concierge_agent],
    instruction="""
    You are a master travel planner who uses data to make recommendations.

    1.  **ALWAYS start with the `call_db_agent` tool** to fetch a list of places (like hotels) that match the user's criteria.

    2.  After you have the data, **use the `call_concierge_agent` tool** to answer any follow-up questions for recommendations, opinions, or advice related to the data you just found.
    """,
    # instruction 明確規定執行順序 → 強制 LLM 走我們設計的 workflow
    # 否則 LLM 可能跳過 db_agent 直接用內部知識回答
)

print(f"✅ Orchestrator Agent '{trip_data_concierge_agent.name}' is defined and ready.")


✅ Orchestrator Agent 'trip_data_concierge' is defined and ready.


In [8]:
# === 測試階層式 Orchestrator ===
# 觀察重點：EVENT log 應該依序出現：
#   1. call_db_agent tool call
#   2. db_agent 回 mock JSON
#   3. call_concierge_agent tool call（state 帶著 db 結果）
#   4. concierge 內部呼叫 food_critic_agent
#   5. food_critic 回 witty 推薦
#   6. concierge 包裝結果
#   7. Orchestrator final_response

async def run_trip_data_concierge():
    """
    建 session 並對頂層 orchestrator 發 query
    """
    concierge_session = await session_service.create_session(
        app_name=trip_data_concierge_agent.name,
        user_id=my_user_id
    )

    # 這個 query 故意設計成「兩段式」：
    # 1. 先要資料（觸發 db_agent）
    # 2. 再要推薦（觸發 concierge → food_critic 鏈）
    query = "Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews."
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(trip_data_concierge_agent, query, concierge_session, my_user_id)

await run_trip_data_concierge()


🗣️ User Query: 'Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews.'

🚀 Running query for agent: 'trip_data_concierge' in session: '30dda6cc-df97-4c0a-a3c5-9671c648a95c'...


/Users/kevinluo/google-agent-ecosystem/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/Users/kevinluo/google-agent-ecosystem/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'Top-rated hotels in San Francisco'
        },
        id='adk-ac20548a-ed68-43eb-9a64-f2c19df0e468',
        name='call_db_agent'
      ),
      thought_signature=b'\n\xfc\x04\x01\x8f=k_\x1bgcb\xd9x\x00\xd5\x08\x18A7\xfdjtC\xbaS\xd4*\xd1\xc35\x1d\xbe\x8fTH\x04\x99\x9d\xb3Y\x81|a\x8c\x85\x16\xff\xd4\xcb\x1d\xe6#\xe3tL\xf1\x99\xc78\xe2\x82\xeb\xa8m\xefe"\xddS\x80\xbb?\x01\xcf\x04`\x8c\xbc\xc4y\x1c\xe3\x9cV\x00\xc1\xfa\xec\x03*j\xc1\x86a\xb9\t...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=13,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=13


/Users/kevinluo/google-agent-ecosystem/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'Suggest a dinner spot near Seaside Inn in San Francisco.'
        },
        id='adk-0bb43ec0-75b4-4749-b9d4-fe8b3fccedde',
        name='call_concierge_agent'
      ),
      thought_signature=b"\n\xa2\x05\x01\x8f=k_kJ\xabyi\xb6[}\x1b3\x82d\x85\xc9\xe3\x0c9\xf9'=\xe5\xa8e\xaf\xea\\\xf2\xca\x97\x11\xe6\xb3\xbd\x07\x83:\xa9mh\xbdL\xeb/\t\xda2\xd6\xfa\x1f\x9a_\xdd\xf6\xcc,\xdf\xe4\x82\x08\x18\xbd\x94\xd9\x12)\xe6\xa0t+d\xe5l\xcc\x88pU\xc1\xe50\xce\x18Y\xa5\rL\x17\x1c\xc7\x1a...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=18,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT

...

EVENT: model_version=None content=Content(
  parts=[
    Part(
      function_response=FunctionResponse(
        id='adk-0bb43ec0-75b4-4749-b9d4-fe8b3fccedde',
        name='call_concierge_agent',
        response={
          'result': 'Certainly, for an exquisite dining experience near the Seaside Inn in San Francisco, I highly recommend Zuni Café. Their roast chicken is particularly renowned and comes with a rather enthusiastic endorsement from our food critic. I trust you will find it to your liking.'
        }
      )
    ),
  ],
  role='user'
) grounding_metadata=None partial=None turn_complete=None finish_reason=None error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=None live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None interaction_id=None invocation_id='e-2191dfe4-2d31-4603-90ff-d119432ab38a' author='trip_data_concierge' act

/Users/kevinluo/google-agent-ecosystem/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""The top-rated hotels in San Francisco are The Grand Hotel (5 stars, 450 reviews) and Seaside Inn (4 stars, 620 reviews).

Seaside Inn has the most reviews. For an exquisite dining experience near the Seaside Inn, I highly recommend Zuni Café. Their roast chicken is particularly renowned and comes with a rather enthusiastic endorsement from our food critic. I trust you will find it to your liking.""",
      thought_signature=b'\n\xfe\x04\x01\x8f=k_\xcb%\xc2\xe8\x86\xdf\xc1\x94\xf1\x9dB=0\x19N{B\x19K\xd8\xba\xc7 \x86\xa36\xda\x88\x80\xaer@)\x99\x0c\xd8[\x85}V\x1c\xb1*\xacf\xab\xa1\x9e\xd9\xde\x16\xa9\x83\xc0\xc9\xdd\xb25\xc2+\x8e`j|\xaebn\x16\xb7\xa6W\xe4s\xdf\xde\xed\xce\xc8\x94d\xdbV\x1e\nVE/v\x11...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=Non

The top-rated hotels in San Francisco are The Grand Hotel (5 stars, 450 reviews) and Seaside Inn (4 stars, 620 reviews).

Seaside Inn has the most reviews. For an exquisite dining experience near the Seaside Inn, I highly recommend Zuni Café. Their roast chicken is particularly renowned and comes with a rather enthusiastic endorsement from our food critic. I trust you will find it to your liking.

--------------------------------------------------



---
## 第 3 部分：具有記憶的 Agent - 自適應規劃師 🗺️

現在，讓我們看看一個不僅能**記住**還能**適應**的 agent。我們將挑戰 `multi_day_trip_agent` 根據我們的回饋重新規劃其行程的一部分。這是對對話式 AI 更為實際的測試。

```
+-----------------------------------------------------+
|         自適應多日行程 Agent 🗺️                     |
|-----------------------------------------------------|
|  模型：gemini-2.5-flash                             |
|  描述：                                             |
|   逐步建立多日旅遊行程，                             |
|   記住前幾天的內容，根據回饋調整                      |
|-----------------------------------------------------|
|  🔧 工具：                                          |
|   - Google Search                                   |
|-----------------------------------------------------|
|  🧠 能力：                                          |
|   - 記住過去的對話和偏好                              |
|   - 漸進式規劃（一次一天）                            |
|   - 根據使用者回饋進行調整                            |
|   - 確保跨天活動的多樣性                              |
+-----------------------------------------------------+

            ▲
            |
    +---------------------------+
    |     使用者互動             |
    |---------------------------|
    | - 目的地                   |
    | - 旅程天數                 |
    | - 興趣與回饋               |
    +---------------------------+

            |
            ▼

+-----------------------------------------------------+
|        逐日行程生成                                  |
|-----------------------------------------------------|
|  🗓️ 第 N 天輸出（Markdown 格式）：                   |
|   - 早上 / 下午 / 晚上活動                           |
|   - 個人化且具有情境感知                              |
|   - 接受變更，確認回饋                                |
+-----------------------------------------------------+

            |
            ▼

+-----------------------------------------------------+
|        觸發下一天的規劃 🚀                           |
|-----------------------------------------------------|
| - 在前幾天的基礎上建構                                |
| - 避免重複                                           |
| - 在繼續之前詢問使用者確認                            |
+-----------------------------------------------------+
```


In [ ]:
# === Agent 定義：Adaptive Multi-Day Trip Planner ===
# 重點：這個 agent「依賴 session 記憶」才能正確運作
# 如果用獨立 session 跑每一輪 → 後面的對話會失憶（cell 33 會示範失敗案例）

def create_multi_day_trip_agent():
    """建立 Progressive Multi-Day Trip Planner agent

    與 cell 12 的 day_trip_agent 差異：
    - day_trip_agent: 一次給完整一日行程（無狀態）
    - multi_day_agent: 一次只規劃一天，等使用者回饋後才規劃下一天（依賴對話歷史）
    """
    return Agent(
        name="multi_day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction="""
        You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.

        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        """,
        # 注意 instruction 中明確強調「short-term memory」「MUST refer back to our conversation」
        # → 提示 LLM 主動引用對話歷史，而不是每次重新開始

        tools=[google_search]
    )

multi_day_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_agent.name}' is created and ready to plan and adapt!")


🗺️ Agent 'multi_day_trip_agent' is created and ready to plan and adapt!


### 情境 3a：具有記憶的 Agent（使用單一 Session）✅

首先，讓我們看看正確的做法。我們將在整個對話中使用**完全相同的 `trip_session` 物件**。觀察 agent 如何記住第 1 輪的上下文，以正確處理第 2 輪和第 3 輪的請求。

In [ ]:
# === DEMO 2: 有記憶的 Agent（重複使用同一個 session）===
# 關鍵設計：三輪對話「全部用同一個 trip_session」
# → ADK 會自動把對話歷史塞進每次 LLM 呼叫的 context
# → agent 看得到前面說過什麼，能做 incremental planning

async def run_adaptive_memory_demonstration():
    print("### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###")

    # 建一個 session，三輪對話共用 ⭐
    trip_session = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"Created a single session for our trip: {trip_session.id}")

    # === Turn 1: 起手式 — 提出旅行需求 ===
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    print(f"\n🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, trip_session, my_user_id)

    # === Turn 2: 給 feedback 並要求修改 ===
    # 注意：用「同一個 trip_session」→ agent 會記得前面提過 Lisbon 與 historic sites
    query2 = "That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?"
    print(f"\n🗣️ User (Turn 2 - Feedback): '{query2}'")
    await run_agent_query(multi_day_agent, query2, trip_session, my_user_id)

    # === Turn 3: 確認並請繼續 ===
    # agent 應該知道「Day 1 已調整過」，現在規劃 Day 2 並延續美食主題
    query3 = "Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind."
    print(f"\n🗣️ User (Turn 3 - Confirmation): '{query3}'")
    await run_agent_query(multi_day_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()


### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###
Created a single session for our trip: 1e5954e1-1a8e-4bdd-9fa3-a737cbca7b31

🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '1e5954e1-1a8e-4bdd-9fa3-a737cbca7b31'...

--------------------------------------------------
✅ Final Response:


An error occurred: name 'Content' is not defined

--------------------------------------------------


🗣️ User (Turn 2 - Feedback): 'That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '1e5954e1-1a8e-4bdd-9fa3-a737cbca7b31'...

--------------------------------------------------
✅ Final Response:


An error occurred: name 'Content' is not defined

--------------------------------------------------


🗣️ User (Turn 3 - Confirmation): 'Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '1e5954e1-1a8e-4bdd-9fa3-a737cbca7b31'...

--------------------------------------------------
✅ Final Response:


An error occurred: name 'Content' is not defined

--------------------------------------------------



### 情境 3b：沒有記憶的 Agent（使用獨立的 Session）❌

現在，讓我們看看如果搞砸了 session 管理會發生什麼。在這裡，我們將透過為每一輪對話建立**全新的、獨立的 session** 來讓 agent 患上失憶症。

請特別注意 agent 對第二個查詢的回應。因為它在一個新的 session 中，它對我們剛才討論的里斯本之旅毫無記憶！

In [ ]:
# === DEMO 2b: 失憶的 Agent（每輪用不同 session）===
# 反例示範：同一個 agent，但每輪對話用「新 session」
# → 每次 LLM 看到的 context 只有當下這一句 query
# → Turn 2 會問「我們在規劃什麼旅行？」（因為它看不到 Turn 1 的內容）
# 這個 demo 強調「session 是 memory 的關鍵」

async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print("### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###")
    print("#"*60)

    # === Turn 1: 第一個 session 內提需求 ===
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    session_one = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a session for Turn 1: {session_one.id}")
    print(f"🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, session_one, my_user_id)

    # === Turn 2: 換一個全新 session 問「請規劃 Day 2」===
    # ⚠️ 這裡用 session_two 而不是 session_one！這就是錯誤示範
    # agent 不知道你在說哪個旅行的 Day 2 → 會困惑、可能反問
    query2 = "Yes, that looks perfect! Please plan Day 2."
    session_two = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a BRAND NEW session for Turn 2: {session_two.id}")
    print(f"🗣️ User (Turn 2): '{query2}'")
    await run_agent_query(multi_day_agent, query2, session_two, my_user_id)

    # 學到什麼？
    # → 想做「多輪對話」一定要重複用同一個 session_id
    # → 想做「獨立查詢」就每次建新 session（避免歷史污染）

await run_memory_failure_demonstration()



############################################################
### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###
############################################################

Created a session for Turn 1: f2af24eb-2bfc-4235-813c-729657daf75d
🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'f2af24eb-2bfc-4235-813c-729657daf75d'...

--------------------------------------------------
✅ Final Response:


An error occurred: name 'Content' is not defined

--------------------------------------------------


Created a BRAND NEW session for Turn 2: 5586ddab-2d07-4cf5-8510-008768d09820
🗣️ User (Turn 2): 'Yes, that looks perfect! Please plan Day 2.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '5586ddab-2d07-4cf5-8510-008768d09820'...

--------------------------------------------------
✅ Final Response:


An error occurred: name 'Content' is not defined

--------------------------------------------------



看到了嗎？Agent 感到困惑了！它很可能詢問了我們在談論什麼目的地或什麼旅行。因為第二個查詢在一個全新的、隔離的 session 中，agent 對規劃里斯本第 1 天的行程毫無記憶。

這完美地說明了為什麼**管理 session 是建構真正對話式 agent 的關鍵！**

---
## 🎉 恭喜！🎉

恭喜你完成了 ADK 工具與記憶的冒險之旅！你已經從建構單次互動的 agent 邁出了巨大的一步，進入了建立動態、有狀態的 AI 系統的領域。

讓我們回顧一下你已經掌握的強大概念：

- **基礎 Agent 與工具**：你從建構「一日遊精靈」開始，並為它配備了第一個工具 GoogleSearch。

- **自訂函式工具**：你透過建立一個從美國國家氣象局 API 取得即時資料的自訂工具，賦予了 agent 新的感知能力。

- **Agent 即工具**：你協調了一個精密的階層結構，其中 agent 將任務委派給其他更專業的 agent，建立了一個協作團隊。

- **記憶的力量**：最重要的是，你親身體驗了管理單一、持久的 Session 如何讓 agent 記住上下文、適應使用者回饋，並進行有意義的多輪對話。

```
   __            /\_/\         /\_/\        /\_/\         __             (\__/)
o-''|\_____/).  ( o.o )       ( -.- )      ( ^_^ )     o-''|\_____/).    ( ^_^ )
 \_/|_)     )    > ^ <         > * <        >💖<         \_/|_)     )     / >🌸< \
    \  __  /                                              \  __  /         /   \
    (_/ (_/                                               (_/ (_/        (___|___)
```
